# 03 — Adversarial Attack

> **Goal:** Show that ML-based IDS can be fooled by small feature perturbations.

We attack a trained Random Forest via a Logistic Regression *surrogate*:
1. Compute gradient of LR loss w.r.t. input
2. Perturb attack samples toward the benign direction
3. Measure how many attack flows are now misclassified as benign

Use the interactive slider below to explore different ε values.

In [ ]:
import os, sys
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## Setup — load data and train models

In [ ]:
from src.data_pipeline import load_data, auto_detect_dataset, get_feature_bounds
from src.model import train_random_forest, evaluate
from src.attacker import train_surrogate, fgsm_attack, evaluate_attack

data_path, _ = auto_detect_dataset()
X_train, X_test, y_train, y_test, scaler = load_data(data_path, n_features=20)
X_min, X_max = get_feature_bounds(X_train)

rf = train_random_forest(X_train, y_train)
surrogate = train_surrogate(X_train, y_train)

baseline = evaluate(rf, X_test, y_test)
print(f'Baseline detection rate: {baseline["detection_rate"]*100:.1f}%')

## FGSM Attack — Epsilon Sweep

We sweep ε from 0.0 to 0.5 and record the detection rate at each step.

In [ ]:
EPSILONS = [0.0, 0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]
detection_rates = []

for eps in EPSILONS:
    X_adv = fgsm_attack(surrogate, X_test, y_test, eps, X_min, X_max)
    dr = evaluate_attack(rf, X_adv, y_test)
    detection_rates.append(dr * 100)
    print(f'  eps={eps:.2f}  detection={dr*100:.1f}%')

### Static plot of the sweep

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(EPSILONS, detection_rates, 'b-o', linewidth=2.5, markersize=9, label='FGSM')
ax.axhline(baseline['detection_rate']*100, color='green', linestyle='--',
           linewidth=2, label=f'Baseline {baseline["detection_rate"]*100:.1f}%')
ax.fill_between(EPSILONS, detection_rates, alpha=0.15, color='blue')
ax.set_xlabel('Epsilon')
ax.set_ylabel('Detection Rate (%)')
ax.set_title('FGSM Attack: Detection Rate vs Epsilon')
ax.legend()
ax.set_ylim(0, 105)
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'plots', 'fgsm_sweep_nb.png'), dpi=120)
plt.show()

## Interactive Epsilon Slider

> Run this cell and drag the slider to see the confusion matrix update live.
> (Requires `ipywidgets` and a live Jupyter kernel — not visible in nbviewer.)

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    from sklearn.metrics import confusion_matrix

    out = widgets.Output()

    def update_attack(epsilon):
        with out:
            clear_output(wait=True)
            X_adv = fgsm_attack(surrogate, X_test, y_test, epsilon, X_min, X_max)
            y_pred = rf.predict(X_adv)
            cm = confusion_matrix(y_test, y_pred)
            dr = evaluate_attack(rf, X_adv, y_test)

            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            # Confusion matrix
            cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
            sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
                        xticklabels=['Benign','Attack'],
                        yticklabels=['Benign','Attack'], ax=axes[0])
            axes[0].set_title(f'Confusion Matrix  ε={epsilon:.2f}\nDetection: {dr*100:.1f}%')
            axes[0].set_xlabel('Predicted')
            axes[0].set_ylabel('Actual')

            # Detection rate bar
            axes[1].bar(['Baseline','Under Attack'],
                        [baseline['detection_rate']*100, dr*100],
                        color=['#2ecc71','#e74c3c'], edgecolor='black')
            axes[1].set_ylabel('Detection Rate (%)')
            axes[1].set_ylim(0, 105)
            axes[1].set_title('Detection Rate Comparison')
            for bar, val in zip(axes[1].patches, [baseline['detection_rate']*100, dr*100]):
                axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                             f'{val:.1f}%', ha='center', fontsize=13, fontweight='bold')

            plt.suptitle(f'FGSM Adversarial Attack  |  ε = {epsilon:.2f}', fontsize=13)
            plt.tight_layout()
            plt.show()

    slider = widgets.FloatSlider(
        value=0.0, min=0.0, max=0.5, step=0.05,
        description='Epsilon ε:',
        continuous_update=False,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='500px')
    )
    interactive = widgets.interactive(update_attack, epsilon=slider)
    display(interactive, out)
    update_attack(0.0)

except ImportError:
    print('ipywidgets not available. Run: pip install ipywidgets')
    print('Static plots are still generated above.')

## Scatter: original vs perturbed samples

Visualise in 2D (first two features after PCA) how attack samples shift after FGSM.

In [ ]:
from sklearn.decomposition import PCA

eps_demo = 0.3
X_adv_demo = fgsm_attack(surrogate, X_test, y_test, eps_demo, X_min, X_max)

attack_mask = y_test == 1
X_attack_orig = X_test[attack_mask]
X_attack_adv = X_adv_demo[attack_mask]
X_benign = X_test[~attack_mask]

# PCA projection for visualisation
pca = PCA(n_components=2, random_state=42)
all_data = np.vstack([X_benign, X_attack_orig, X_attack_adv])
projected = pca.fit_transform(all_data)

n_b = len(X_benign)
n_a = len(X_attack_orig)
p_benign = projected[:n_b]
p_attack_orig = projected[n_b:n_b+n_a]
p_attack_adv = projected[n_b+n_a:]

fig, ax = plt.subplots(figsize=(9, 7))
ax.scatter(p_benign[:,0], p_benign[:,1], c='#2ecc71', alpha=0.4, s=25, label='Benign')
ax.scatter(p_attack_orig[:,0], p_attack_orig[:,1], c='#e74c3c', alpha=0.6, s=30, label='Attack (original)')
ax.scatter(p_attack_adv[:,0], p_attack_adv[:,1], c='#f39c12', alpha=0.6, s=30,
           label=f'Attack (after FGSM ε={eps_demo})', marker='^')

# Draw arrows for a sample of shifts
for i in range(0, min(30, n_a), 3):
    ax.annotate('', xy=(p_attack_adv[i,0], p_attack_adv[i,1]),
                xytext=(p_attack_orig[i,0], p_attack_orig[i,1]),
                arrowprops=dict(arrowstyle='->', color='gray', alpha=0.5, lw=0.8))

ax.set_xlabel('PCA Component 1')
ax.set_ylabel('PCA Component 2')
ax.set_title(f'Adversarial Perturbation in Feature Space (ε={eps_demo})')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join('..', 'results', 'plots', 'perturbation_scatter.png'), dpi=120, bbox_inches='tight')
plt.show()

print(f'Arrows show how attack samples move toward benign region after FGSM.')